# 4_1_io_model_comparison_large

2020-2025 大样本模型横向比较入口。

- 保留 `4_io_model_comparison_ioexp.ipynb` 作为小样本快速验证。
- 本 notebook 支持 `speed / balance / accuracy` 三档，并提供窗口级缓存与断点续跑。

In [1]:
import os
import sys
import time
import pickle
from dataclasses import replace
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.append('../models')

import ioexp_runner as io_runner_module
from ioexp_baselines import price_bs_flat, price_heston, price_localvol_quadratic, price_sabr
from ioexp_data_contract import IOSliceConfig, generate_rolling_slices, select_slice
from ioexp_eval_metrics import compute_group_metrics
from ioexp_runner import IOExperimentRunner


In [2]:
# -----------------------------
# 运行模式与输入路径
# -----------------------------
MODE = 'turbo'  # 'turbo' | 'speed' | 'balance' | 'accuracy'

RAW_CSV = Path('../data/io_option_cross_section_20200101_20231231.csv')
if not RAW_CSV.exists():
    # 当前仓库若暂无全样本文件，先回退到小样本文件做流程验证
    RAW_CSV = Path('../data/io_option_cross_section_20240101_20240131.csv')

RFSV_PATH = Path('../data/rfsv_predictions_20200101_20251231.csv')
RFSV_ARG = str(RFSV_PATH) if RFSV_PATH.exists() else None

CACHE_ROOT = Path('../data/ioexp_large_cache')
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

print('MODE =', MODE)
print('RAW_CSV =', RAW_CSV)
print('RFSV_ARG =', RFSV_ARG)

MODE = turbo
RAW_CSV = ../data/io_option_cross_section_20200101_20231231.csv
RFSV_ARG = None


In [3]:
# -----------------------------
# 并行运行配置（建议先用默认值）
# -----------------------------
CPU_TOTAL = os.cpu_count() or 8
N_WORKERS = max(1, min(CPU_TOTAL - 2, 6), 6)
MAX_INFLIGHT = max(N_WORKERS, 2 * N_WORKERS)
FORCE_SEQUENTIAL_DEBUG = False

# 避免多进程 + BLAS 多线程争抢资源
for _k in [
    'OMP_NUM_THREADS',
    'MKL_NUM_THREADS',
    'OPENBLAS_NUM_THREADS',
    'NUMEXPR_NUM_THREADS',
    'VECLIB_MAXIMUM_THREADS',
]:
    os.environ.setdefault(_k, '1')

print('CPU_TOTAL =', CPU_TOTAL)
print('N_WORKERS =', N_WORKERS)
print('MAX_INFLIGHT =', MAX_INFLIGHT)
print('FORCE_SEQUENTIAL_DEBUG =', FORCE_SEQUENTIAL_DEBUG)

CPU_TOTAL = 12
N_WORKERS = 6
MAX_INFLIGHT = 12
FORCE_SEQUENTIAL_DEBUG = False


In [4]:
# -----------------------------
# 可选：自动抓取 2020-2023 IO 期权截面（带库存检查）
# -----------------------------
FORCE_REFETCH = True
RAW_CSV = Path('../data/io_option_cross_section_20200101_20231231.csv')

if RAW_CSV.exists() and not FORCE_REFETCH:
    print(f'Found existing dataset, skip fetching: {RAW_CSV}')
else:
    from OptionDataFetcher import OptionDataFetcher
    from dotenv import load_dotenv
    load_dotenv()

    RAW_CSV.parent.mkdir(parents=True, exist_ok=True)
    fetcher = OptionDataFetcher()
    fetcher.init_connection(os.environ.get('RQDATAC_LICENSE') or os.environ.get('RQ_API_KEY'))

    _ = fetcher.fetch_option_panel_for_underlying(
        underlying_order_book_id='000300.XSHG',
        start_date='20200101',
        end_date='20231231',
        risk_free_rate=0.02,
        dividend_yield=0.0,
        apply_cleaning=True,
        save_path=str(RAW_CSV),
        include_api_iv=False,
        verbose=True,
        min_days_to_expiry=5,
        min_volume=0.0,
        max_abs_log_moneyness=None,
        contract_snapshot_step=10
    )
    print(f'Fetch done. saved to: {RAW_CSV}')

✓ 米筐数据连接成功
[OptionDataFetcher] step1/4 获取时间段合约池
[OptionDataFetcher] step2/4 批量拉取期权价格: 2936 个合约 (snapshot_days=98, step=10)
[OptionDataFetcher] step3/4 拉取标的数据并合并
✓ 已保存期权面板数据: ../data/io_option_cross_section_20200101_20231231.csv
✓ 期权面板完成: 227514 行, 覆盖 970 个交易日
Fetch done. saved to: ../data/io_option_cross_section_20200101_20231231.csv


In [5]:
# -----------------------------
# 三档配置（速度/平衡/精度）
# -----------------------------
PROFILE = {
    'turbo': {
        'data_cfg': IOSliceConfig(train_days=40, valid_days=10, test_days=10, step_days=10, min_total_days=60),
        'mc_stage1_paths': 1000,
        'mc_stage2_paths': 3000,
        'maxiter_stage1': 10,
        'maxiter_stage2': 16,
        'baseline_models': ['bs_flat', 'localvol_quadratic_iv'],
    },
    'speed': {
        'data_cfg': IOSliceConfig(train_days=126, valid_days=21, test_days=21, step_days=21, min_total_days=168),
        'mc_stage1_paths': 3000,
        'mc_stage2_paths': 8000,
        'maxiter_stage1': 12,
        'maxiter_stage2': 20,
        'baseline_models': ['bs_flat', 'localvol_quadratic_iv'],
    },
    'balance': {
        'data_cfg': IOSliceConfig(train_days=252, valid_days=42, test_days=42, step_days=21, min_total_days=336),
        'mc_stage1_paths': 6000,
        'mc_stage2_paths': 16000,
        'maxiter_stage1': 20,
        'maxiter_stage2': 35,
        'baseline_models': ['bs_flat', 'localvol_quadratic_iv', 'sabr_hagan'],
    },
    'accuracy': {
        'data_cfg': IOSliceConfig(train_days=252, valid_days=63, test_days=63, step_days=21, min_total_days=420),
        'mc_stage1_paths': 8000,
        'mc_stage2_paths': 24000,
        'maxiter_stage1': 30,
        'maxiter_stage2': 50,
        'baseline_models': ['bs_flat', 'localvol_quadratic_iv', 'sabr_hagan', 'heston_cf'],
    },
}

cfg_p = PROFILE[MODE]
BASELINE_FUNCS = {
    'bs_flat': price_bs_flat,
    'localvol_quadratic_iv': price_localvol_quadratic,
    'sabr_hagan': price_sabr,
    'heston_cf': price_heston,
}

print('selected baselines:', cfg_p['baseline_models'])
print('data_cfg:', cfg_p['data_cfg'])

selected baselines: ['bs_flat', 'localvol_quadratic_iv']
data_cfg: IOSliceConfig(train_days=40, valid_days=10, test_days=10, step_days=10, min_total_days=60)


In [6]:
# -----------------------------
# 覆盖 runner 内 baseline 调度（仅在本 notebook 会话中生效）
# -----------------------------
def run_selected_baselines(option_df: pd.DataFrame):
    out = {}
    for name in cfg_p['baseline_models']:
        out[name] = BASELINE_FUNCS[name](option_df)
    return out

io_runner_module.run_all_baselines = run_selected_baselines
print('patched ioexp_runner.run_all_baselines ->', cfg_p['baseline_models'])

patched ioexp_runner.run_all_baselines -> ['bs_flat', 'localvol_quadratic_iv']


In [7]:
# -----------------------------
# Numba 内核（可选）
# -----------------------------
NUMBA_ENABLE = True
NUMBA_READY = False

try:
    from numba import njit
except Exception:
    njit = None

if NUMBA_ENABLE and njit is not None:
    @njit(cache=True)
    def _mc_price_cv_numba(s_terminal, K, T, r, q, S0, is_call, use_control_variate):
        n = s_terminal.shape[0]
        sign = 1.0 if is_call else -1.0
        discount = np.exp(-r * T)

        payoff_sum = 0.0
        payoff_sq_sum = 0.0
        control_sum = 0.0
        control_sq_sum = 0.0
        cross_sum = 0.0

        for i in range(n):
            intrinsic = sign * (s_terminal[i] - K)
            payoff = intrinsic if intrinsic > 0.0 else 0.0
            payoff_d = discount * payoff
            control = discount * s_terminal[i]

            payoff_sum += payoff_d
            payoff_sq_sum += payoff_d * payoff_d
            control_sum += control
            control_sq_sum += control * control
            cross_sum += payoff_d * control

        mean_payoff = payoff_sum / n
        if n > 1:
            var_payoff = (payoff_sq_sum - n * mean_payoff * mean_payoff) / (n - 1)
            if var_payoff < 0.0:
                var_payoff = 0.0
            stderr_raw = np.sqrt(var_payoff) / np.sqrt(n)
        else:
            stderr_raw = 0.0

        if not use_control_variate:
            return mean_payoff, stderr_raw

        mean_control = control_sum / n
        if n <= 1:
            return mean_payoff, stderr_raw

        var_control = (control_sq_sum - n * mean_control * mean_control) / (n - 1)
        if var_control <= 1e-14:
            return mean_payoff, stderr_raw

        cov_pc = (cross_sum - n * mean_payoff * mean_control) / (n - 1)
        beta = cov_pc / var_control
        control_mean_theory = S0 * np.exp(-q * T)

        adj_sum = 0.0
        adj_sq_sum = 0.0
        for i in range(n):
            intrinsic = sign * (s_terminal[i] - K)
            payoff = intrinsic if intrinsic > 0.0 else 0.0
            payoff_d = discount * payoff
            control = discount * s_terminal[i]
            adj = payoff_d - beta * (control - control_mean_theory)
            adj_sum += adj
            adj_sq_sum += adj * adj

        mean_adj = adj_sum / n
        var_adj = (adj_sq_sum - n * mean_adj * mean_adj) / (n - 1)
        if var_adj < 0.0:
            var_adj = 0.0
        stderr_adj = np.sqrt(var_adj) / np.sqrt(n)
        return mean_adj, stderr_adj

    _ = _mc_price_cv_numba(
        np.array([90.0, 100.0, 110.0], dtype=np.float64),
        100.0,
        0.25,
        0.02,
        0.0,
        100.0,
        True,
        True,
    )
    NUMBA_READY = True

print('NUMBA_ENABLE =', NUMBA_ENABLE)
print('NUMBA_READY =', NUMBA_READY)

NUMBA_ENABLE = True
NUMBA_READY = True


In [8]:
# -----------------------------
# 会话级 monkey patch：优先走 Numba payoff 内核
# -----------------------------
from ioexp_rbergomi_pricer import IOExpRBergomiPricer

_ORIG_MC_PRICE_FROM_TERMINAL = IOExpRBergomiPricer.mc_price_from_terminal


def _patched_mc_price_from_terminal(self, s_terminal, K, T, r, q, S0, option_type):
    if NUMBA_READY:
        try:
            s_arr = np.asarray(s_terminal, dtype=np.float64)
            is_call = str(option_type).lower().strip() in {'call', 'c'}
            price, stderr = _mc_price_cv_numba(
                s_arr,
                float(K),
                float(T),
                float(r),
                float(q),
                float(S0),
                bool(is_call),
                bool(self.config.use_control_variate),
            )
            return {'price': float(price), 'stderr': float(stderr)}
        except Exception:
            pass

    return _ORIG_MC_PRICE_FROM_TERMINAL(self, s_terminal, K, T, r, q, S0, option_type)


IOExpRBergomiPricer.mc_price_from_terminal = _patched_mc_price_from_terminal
print('patched IOExpRBergomiPricer.mc_price_from_terminal (session only)')

patched IOExpRBergomiPricer.mc_price_from_terminal (session only)


In [9]:
# =============================================
# 加速优化 1: 缓存 Volterra 过程
# fix_H=True 时，同一 calibration stage 内
# dW1 / y / z 只取决于 (H, T, seed)，不随 eta/rho 变化。
# 缓存后，每个 (H, T, seed) 只算一次卷积。
# =============================================
from scipy.signal import fftconvolve

_ORIG_SIMULATE_PATHS = IOExpRBergomiPricer.simulate_paths
_ORIG_BUILD_VOLTERRA = IOExpRBergomiPricer._build_volterra_process

_sim_cache = {}
_SIM_CACHE_MAX = 256


def _cached_simulate_paths(
    self, S0, T, H, eta, rho, xi=0.04, r=0.0, q=0.0, random_seed=None,
):
    if T <= 0 or eta <= 0 or abs(rho) >= 1 or S0 <= 0:
        return _ORIG_SIMULATE_PATHS(self, S0, T, H, eta, rho, xi, r, q, random_seed)

    n_steps = max(1, int(np.ceil(self.config.n_steps_per_year * T)))
    n_paths = int(self.config.n_paths)
    dt = T / n_steps
    time_grid = np.linspace(0.0, T, n_steps + 1)

    cache_key = (H, T, n_paths, n_steps, self.config.n_steps_per_year,
                 self.config.antithetic, random_seed)

    if cache_key in _sim_cache:
        dW1, y, z = _sim_cache[cache_key]
    else:
        rng = self._make_rng(random_seed)
        dW1 = self._sample_hybrid_increments(n_paths, n_steps, H, rng)
        y = self._build_volterra_process(dW1, H, n_steps)
        z = self._sample_std_normal(n_paths, n_steps, rng) * np.sqrt(dt)

        if len(_sim_cache) >= _SIM_CACHE_MAX:
            _sim_cache.pop(next(iter(_sim_cache)))
        _sim_cache[cache_key] = (dW1, y, z)

    xi_curve = self._build_xi_curve(time_grid, xi)
    v = xi_curve[None, :] * np.exp(
        eta * y - 0.5 * (eta ** 2) * (time_grid[None, :] ** (2.0 * H))
    )

    dB = rho * dW1[:, :, 0] + np.sqrt(1.0 - rho ** 2) * z

    drift = (r - q) * dt
    increments = (
        np.sqrt(np.maximum(v[:, :-1], self.config.floor_variance)) * dB
        + drift
        - 0.5 * v[:, :-1] * dt
    )
    log_s = np.zeros((n_paths, n_steps + 1))
    log_s[:, 1:] = np.cumsum(increments, axis=1)
    s = S0 * np.exp(log_s)
    return {"t": time_grid, "S": s, "V": v}


IOExpRBergomiPricer.simulate_paths = _cached_simulate_paths
print('patched simulate_paths with Volterra cache (session only)')
print(f'_SIM_CACHE_MAX = {_SIM_CACHE_MAX}')

patched simulate_paths with Volterra cache (session only)
_SIM_CACHE_MAX = 256


In [10]:
# =============================================
# 加速优化 2: FFT 批量卷积
# 原始 _build_volterra_process 对每条路径逐个 np.convolve，
# 改为 scipy.signal.fftconvolve 一次性批量处理。
# =============================================
from scipy.signal import fftconvolve as _fftconv

_ORIG_BUILD_VOLTERRA = IOExpRBergomiPricer._build_volterra_process


def _fft_build_volterra_process(self, dW1, H, n_steps):
    alpha = self._alpha(H)
    n_paths = dW1.shape[0]
    y1 = np.zeros((n_paths, n_steps + 1))
    y1[:, 1:] = dW1[:, :, 1]

    g = np.zeros(n_steps + 1)
    for k in range(2, n_steps + 1):
        g[k] = self._g(self._b(k, alpha) / self.config.n_steps_per_year, alpha)

    x = dW1[:, :, 0]  # (n_paths, n_steps)
    gx_full = _fftconv(x, g[np.newaxis, :], mode='full', axes=1)
    y2 = gx_full[:, :n_steps + 1]

    scale = np.sqrt(2.0 * alpha + 1.0)
    return scale * (y1 + y2)


IOExpRBergomiPricer._build_volterra_process = _fft_build_volterra_process
print('patched _build_volterra_process with FFT batch convolution')

patched _build_volterra_process with FFT batch convolution


In [11]:
# =============================================
# 加速优化 3+4: 向量化 IV + market_iv 缓存
# - 用向量化 Newton 迭代替换逐个 brentq（主要用于 model_iv）
# - market_iv 按行 key 缓存，避免优化迭代中重复反解
# =============================================
from scipy.stats import norm

_ORIG_IMPLIED_VOL_BLACK = IOExpRBergomiPricer.implied_vol_black
_ORIG_PRICE_CROSS_SECTION = IOExpRBergomiPricer.price_cross_section

_market_iv_cache = {}


def _implied_vol_black_vec(price, S0, K, T, r, q, option_type):
    price = np.asarray(price, dtype=float)
    S0 = np.asarray(S0, dtype=float)
    K = np.asarray(K, dtype=float)
    T = np.asarray(T, dtype=float)
    r = np.asarray(r, dtype=float)
    q = np.asarray(q, dtype=float)
    option_type = np.asarray(option_type)

    n = price.shape[0]
    out = np.full(n, np.nan, dtype=float)

    valid = (T > 0.0) & (price > 0.0) & (S0 > 0.0) & (K > 0.0)
    if not np.any(valid):
        return out

    w = np.where(np.char.lower(option_type.astype(str)) == 'put', -1.0, 1.0)

    Sv = S0[valid]
    Kv = K[valid]
    Tv = T[valid]
    rv = r[valid]
    qv = q[valid]
    pv = price[valid]
    wv = w[valid]

    disc = np.exp(-rv * Tv)
    sqrtT = np.sqrt(Tv)
    F = Sv * np.exp((rv - qv) * Tv)

    intrinsic = disc * np.maximum(wv * (F - Kv), 0.0)
    target = np.maximum(pv, intrinsic + 1e-12)

    vol = np.full_like(target, 0.25)
    vol = np.clip(vol, 1e-6, 5.0)

    for _ in range(8):
        std = np.maximum(vol * sqrtT, 1e-12)
        d1 = np.log(np.maximum(F, 1e-16) / np.maximum(Kv, 1e-16)) / std + 0.5 * std
        d2 = d1 - std

        nd1 = norm.cdf(wv * d1)
        nd2 = norm.cdf(wv * d2)
        bs = disc * (wv * F * nd1 - wv * Kv * nd2)

        vega = disc * F * norm.pdf(d1) * sqrtT
        step = (bs - target) / np.maximum(vega, 1e-8)
        vol = np.clip(vol - step, 1e-6, 5.0)

    out[valid] = vol
    return out


def _patched_price_cross_section(self, option_df, H, eta, rho, xi=0.04, random_seed=None):
    required = {'underlying', 'r', 'q', 'maturity', 'strike', 'option_type', 'market_price'}
    missing = required.difference(option_df.columns)
    if missing:
        raise ValueError(f'Missing required columns: {sorted(missing)}')

    out = option_df.copy().reset_index(drop=True)
    out['model_price'] = np.nan
    out['model_stderr'] = np.nan
    out['model_iv'] = np.nan
    out['market_iv'] = np.nan

    # market_iv 缓存：同一条报价在优化迭代中只反解一次
    market_iv_vals = np.empty(len(out), dtype=float)
    for i, row in out.iterrows():
        key = (
            round(float(row['underlying']), 8),
            round(float(row['strike']), 8),
            round(float(row['maturity']), 8),
            round(float(row['r']), 8),
            round(float(row['q']), 8),
            str(row['option_type']).lower(),
            round(float(row['market_price']), 8),
        )
        cached = _market_iv_cache.get(key)
        if cached is None:
            cached = _ORIG_IMPLIED_VOL_BLACK(
                float(row['market_price']),
                float(row['underlying']),
                float(row['strike']),
                float(row['maturity']),
                float(row['r']),
                float(row['q']),
                str(row['option_type']),
            )
            _market_iv_cache[key] = float(cached) if np.isfinite(cached) else np.nan
        market_iv_vals[i] = _market_iv_cache[key]

    out['market_iv'] = market_iv_vals

    for iT, T in enumerate(np.sort(out['maturity'].unique())):
        sub_idx = out.index[out['maturity'] == T].to_numpy()
        row0 = out.loc[sub_idx[0]]
        S0 = float(row0['underlying'])
        r = float(row0['r'])
        q = float(row0['q'])

        sim = self.simulate_paths(
            S0=S0,
            T=float(T),
            H=H,
            eta=eta,
            rho=rho,
            xi=xi,
            r=r,
            q=q,
            random_seed=None if random_seed is None else int(random_seed + iT),
        )
        s_terminal = sim['S'][:, -1]

        sub = out.loc[sub_idx]
        K_arr = sub['strike'].to_numpy(dtype=float)
        opt_arr = sub['option_type'].astype(str).to_numpy()

        model_price_arr = np.empty(len(sub_idx), dtype=float)
        model_stderr_arr = np.empty(len(sub_idx), dtype=float)

        for j in range(len(sub_idx)):
            pr = self.mc_price_from_terminal(
                s_terminal=s_terminal,
                K=float(K_arr[j]),
                T=float(T),
                r=r,
                q=q,
                S0=S0,
                option_type=str(opt_arr[j]),
            )
            model_price_arr[j] = float(pr['price'])
            model_stderr_arr[j] = float(pr['stderr'])

        model_iv_arr = _implied_vol_black_vec(
            price=model_price_arr,
            S0=np.full(len(sub_idx), S0, dtype=float),
            K=K_arr,
            T=np.full(len(sub_idx), float(T), dtype=float),
            r=np.full(len(sub_idx), r, dtype=float),
            q=np.full(len(sub_idx), q, dtype=float),
            option_type=opt_arr,
        )

        out.loc[sub_idx, 'model_price'] = model_price_arr
        out.loc[sub_idx, 'model_stderr'] = model_stderr_arr
        out.loc[sub_idx, 'model_iv'] = model_iv_arr

    out['price_error'] = out['model_price'] - out['market_price']
    out['iv_error'] = out['model_iv'] - out['market_iv']
    forward = out['underlying'] * np.exp((out['r'] - out['q']) * out['maturity'])
    out['log_moneyness'] = np.log(out['strike'] / forward)
    return out


IOExpRBergomiPricer.price_cross_section = _patched_price_cross_section
print('patched price_cross_section with vectorized model_iv + cached market_iv')
print('market_iv_cache size =', len(_market_iv_cache))

patched price_cross_section with vectorized model_iv + cached market_iv
market_iv_cache size = 0


In [12]:
# -----------------------------
# 读取数据并构建 runner（按档位覆盖 MC 与校准迭代）
# -----------------------------
raw_df = pd.read_csv(RAW_CSV)

base_runner = IOExperimentRunner()
mc_steps = 126 if MODE in {'turbo', 'speed'} else 252
cfg = replace(
    base_runner.cfg,
    data=cfg_p['data_cfg'],
    mc_stage1=replace(
        base_runner.cfg.mc_stage1,
        n_paths=cfg_p['mc_stage1_paths'],
        n_steps_per_year=mc_steps,
    ),
    mc_stage2=replace(
        base_runner.cfg.mc_stage2,
        n_paths=cfg_p['mc_stage2_paths'],
        n_steps_per_year=mc_steps,
    ),
    calib=replace(
        base_runner.cfg.calib,
        maxiter_stage1=cfg_p['maxiter_stage1'],
        maxiter_stage2=cfg_p['maxiter_stage2'],
    ),
)
runner = IOExperimentRunner(cfg=cfg)
prepared = runner.prepare_dataset(raw_df)

print('mc_steps_per_year =', mc_steps)
print('prepared rows:', len(prepared), 'unique trade days:', prepared['trade_date'].nunique())

mc_steps_per_year = 126
prepared rows: 196829 unique trade days: 970


In [13]:
# -----------------------------
# 生成窗口；若数据太短则自动降级为 demo 小窗口
# -----------------------------
windows = generate_rolling_slices(prepared, config=cfg.data)

if len(windows) == 0:
    demo_cfg = IOSliceConfig(train_days=10, valid_days=5, test_days=5, step_days=5, min_total_days=20)
    windows = generate_rolling_slices(prepared, config=demo_cfg)
    active_data_cfg = demo_cfg
    print('No windows under selected mode; fallback to demo cfg:', active_data_cfg)
else:
    active_data_cfg = cfg.data

print('n_windows =', len(windows))
windows[:2] if len(windows) else []

n_windows = 92


[{'train_start': Timestamp('2020-01-02 00:00:00'),
  'train_end': Timestamp('2020-03-05 00:00:00'),
  'valid_start': Timestamp('2020-03-06 00:00:00'),
  'valid_end': Timestamp('2020-03-19 00:00:00'),
  'test_start': Timestamp('2020-03-20 00:00:00'),
  'test_end': Timestamp('2020-04-02 00:00:00')},
 {'train_start': Timestamp('2020-01-16 00:00:00'),
  'train_end': Timestamp('2020-03-19 00:00:00'),
  'valid_start': Timestamp('2020-03-20 00:00:00'),
  'valid_end': Timestamp('2020-04-02 00:00:00'),
  'test_start': Timestamp('2020-04-03 00:00:00'),
  'test_end': Timestamp('2020-04-17 00:00:00')}]

In [ ]:
# -----------------------------
# 窗口级断点续跑：多进程并行（每个窗口独立缓存）
# -----------------------------
cache_dir = CACHE_ROOT / f"{MODE}_windows"
cache_dir.mkdir(parents=True, exist_ok=True)

summary_rows = []
group_rows = []
window_perf_rows = []

horizon = active_data_cfg.train_days + active_data_cfg.valid_days + active_data_cfg.test_days
single_cfg = replace(
    cfg,
    data=IOSliceConfig(
        train_days=active_data_cfg.train_days,
        valid_days=active_data_cfg.valid_days,
        test_days=active_data_cfg.test_days,
        step_days=horizon,
        min_total_days=horizon,
    ),
)


def _run_window_worker(i: int, win: dict):
    t_start = time.time()
    test_start = pd.Timestamp(win['test_start']).strftime('%Y%m%d')
    test_end = pd.Timestamp(win['test_end']).strftime('%Y%m%d')
    cache_file = cache_dir / f'window_{i:04d}_{test_start}_{test_end}.pkl'

    if cache_file.exists():
        with open(cache_file, 'rb') as f:
            res = pickle.load(f)
        loaded = True
    else:
        sub_df = select_slice(prepared, pd.Timestamp(win['train_start']), pd.Timestamp(win['test_end']))
        sub_runner = IOExperimentRunner(cfg=single_cfg)
        # 每个窗口使用稳定、可复现的种子偏移
        sub_runner.calibrator.random_seed = int(1234 + i * 1000)
        res = sub_runner.run_single_test_window(
            prepared_df=sub_df,
            rfsv_pred_path=RFSV_ARG,
            H_init=0.1,
            eta_init=1.0,
            rho_init=-0.7,
        )
        with open(cache_file, 'wb') as f:
            pickle.dump(res, f)
        loaded = False

    s = res['summary'].copy()
    s['window_id'] = i
    s['test_start'] = pd.Timestamp(win['test_start'])
    s['test_end'] = pd.Timestamp(win['test_end'])
    s['loaded_from_cache'] = loaded

    gframes = []
    rb = res['rbergomi_test_df_with_rfsv_prior']
    if rb is not None and not rb.empty and 'maturity_bucket' in rb.columns:
        gm = compute_group_metrics(rb, 'maturity_bucket')
        gm['window_id'] = i
        gm['test_start'] = pd.Timestamp(win['test_start'])
        gm['test_end'] = pd.Timestamp(win['test_end'])
        gm['group_type'] = 'maturity_bucket'
        gframes.append(gm)

    if rb is not None and not rb.empty and 'moneyness_bucket' in rb.columns:
        gk = compute_group_metrics(rb, 'moneyness_bucket')
        gk['window_id'] = i
        gk['test_start'] = pd.Timestamp(win['test_start'])
        gk['test_end'] = pd.Timestamp(win['test_end'])
        gk['group_type'] = 'moneyness_bucket'
        gframes.append(gk)

    elapsed_sec = time.time() - t_start
    return {'summary': s, 'group_frames': gframes, 'loaded': loaded, 'window_id': i, 'elapsed_sec': elapsed_sec}


def _collect_result(payload: dict):
    summary_rows.append(payload['summary'])
    for _g in payload['group_frames']:
        group_rows.append(_g)
    window_perf_rows.append(
        {
            'window_id': int(payload['window_id']),
            'elapsed_sec': float(payload['elapsed_sec']),
            'loaded_from_cache': bool(payload['loaded']),
        }
    )


def _render_progress(done: int, total: int, t0_: float):
    total = max(total, 1)
    pct = done / total
    elapsed_s = time.time() - t0_
    eta_s = (elapsed_s / done) * (total - done) if done > 0 else 0.0
    bar_len = 24
    filled = int(bar_len * pct)
    bar = '#' * filled + '-' * (bar_len - filled)
    print(
        f"\r[{bar}] {done:>3}/{total:<3} {pct*100:6.2f}% | elapsed={elapsed_s/60:.2f}m | eta={eta_s/60:.2f}m",
        end='',
        flush=True,
    )


t0 = time.time()
tasks = [(i, win) for i, win in enumerate(windows, 1)]
total_tasks = len(tasks)

if FORCE_SEQUENTIAL_DEBUG or N_WORKERS <= 1:
    print('Running sequential mode ...')
    done = 0
    _render_progress(done, total_tasks, t0)
    for i, win in tasks:
        payload = _run_window_worker(i, win)
        _collect_result(payload)
        done += 1
        _render_progress(done, total_tasks, t0)
    print()
else:
    print(f'Running parallel mode with N_WORKERS={N_WORKERS}, MAX_INFLIGHT={MAX_INFLIGHT}')
    import multiprocessing as mp

    try:
        ctx = mp.get_context('fork')
    except ValueError:
        ctx = mp.get_context('spawn')

    done = 0
    _render_progress(done, total_tasks, t0)
    with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=ctx) as ex:
        pending = {}
        task_ptr = 0

        while task_ptr < len(tasks) and len(pending) < MAX_INFLIGHT:
            i, win = tasks[task_ptr]
            fut = ex.submit(_run_window_worker, i, win)
            pending[fut] = i
            task_ptr += 1

        while pending:
            fut = next(as_completed(list(pending.keys())))
            _ = pending.pop(fut)
            payload = fut.result()
            _collect_result(payload)
            done += 1
            _render_progress(done, total_tasks, t0)

            while task_ptr < len(tasks) and len(pending) < MAX_INFLIGHT:
                i, win = tasks[task_ptr]
                nfut = ex.submit(_run_window_worker, i, win)
                pending[nfut] = i
                task_ptr += 1
    print()

elapsed_min = (time.time() - t0) / 60.0
window_perf_df = pd.DataFrame(window_perf_rows).sort_values('window_id').reset_index(drop=True)
print(f'All windows done. elapsed={elapsed_min:.2f} min')
if not window_perf_df.empty:
    cache_hit = window_perf_df['loaded_from_cache'].mean()
    print(f'avg_window_sec={window_perf_df["elapsed_sec"].mean():.2f}, cache_hit_rate={cache_hit:.2%}')

Running parallel mode with N_WORKERS=6, MAX_INFLIGHT=12
[#######-----------------]  28/92   30.43% | elapsed=35.33m | eta=80.75mm

In [ ]:
# -----------------------------
# A/B 一致性检查（可选，默认关闭）
# -----------------------------
RUN_AB_VALIDATION = False
AB_WINDOW_COUNT = 1
AB_TOL_IV_RMSE = 0.01
AB_TOL_IV_MAE = 0.01
AB_TOL_PRICE_RMSE = 0.05

ab_compare_df = pd.DataFrame()

if not RUN_AB_VALIDATION:
    print('A/B validation skipped (set RUN_AB_VALIDATION=True to enable).')
else:
    if len(windows) == 0:
        print('No windows available for A/B validation.')
    else:
        use_n = max(1, min(int(AB_WINDOW_COUNT), len(windows)))
        test_windows = [(i, windows[i - 1]) for i in range(1, use_n + 1)]

        # 原始实现（关闭 Numba patch）
        IOExpRBergomiPricer.mc_price_from_terminal = _ORIG_MC_PRICE_FROM_TERMINAL
        rows_a = []
        for i, win in test_windows:
            sub_df = select_slice(prepared, pd.Timestamp(win['train_start']), pd.Timestamp(win['test_end']))
            r0 = IOExperimentRunner(cfg=single_cfg)
            r0.calibrator.random_seed = int(1234 + i * 1000)
            out0 = r0.run_single_test_window(
                prepared_df=sub_df,
                rfsv_pred_path=RFSV_ARG,
                H_init=0.1,
                eta_init=1.0,
                rho_init=-0.7,
            )
            s0 = out0['summary'].copy()
            s0['window_id'] = i
            rows_a.append(s0)

        # 新实现（Numba patch）
        IOExpRBergomiPricer.mc_price_from_terminal = _patched_mc_price_from_terminal
        rows_b = []
        for i, win in test_windows:
            sub_df = select_slice(prepared, pd.Timestamp(win['train_start']), pd.Timestamp(win['test_end']))
            r1 = IOExperimentRunner(cfg=single_cfg)
            r1.calibrator.random_seed = int(1234 + i * 1000)
            out1 = r1.run_single_test_window(
                prepared_df=sub_df,
                rfsv_pred_path=RFSV_ARG,
                H_init=0.1,
                eta_init=1.0,
                rho_init=-0.7,
            )
            s1 = out1['summary'].copy()
            s1['window_id'] = i
            rows_b.append(s1)

        A = pd.concat(rows_a, ignore_index=True)
        B = pd.concat(rows_b, ignore_index=True)

        kcols = ['window_id', 'model']
        mcols = ['iv_rmse', 'iv_mae', 'price_rmse']
        ab_compare_df = A[kcols + mcols].merge(B[kcols + mcols], on=kcols, suffixes=('_orig', '_new'))

        for m in mcols:
            ab_compare_df[f'delta_{m}'] = ab_compare_df[f'{m}_new'] - ab_compare_df[f'{m}_orig']
            ab_compare_df[f'abs_delta_{m}'] = ab_compare_df[f'delta_{m}'].abs()

        pass_iv_rmse = bool((ab_compare_df['abs_delta_iv_rmse'] <= AB_TOL_IV_RMSE).all())
        pass_iv_mae = bool((ab_compare_df['abs_delta_iv_mae'] <= AB_TOL_IV_MAE).all())
        pass_price_rmse = bool((ab_compare_df['abs_delta_price_rmse'] <= AB_TOL_PRICE_RMSE).all())

        print('A/B rows =', len(ab_compare_df))
        print('A/B pass iv_rmse:', pass_iv_rmse, 'tol=', AB_TOL_IV_RMSE)
        print('A/B pass iv_mae:', pass_iv_mae, 'tol=', AB_TOL_IV_MAE)
        print('A/B pass price_rmse:', pass_price_rmse, 'tol=', AB_TOL_PRICE_RMSE)
        ab_compare_df.head(20)

In [ ]:
# -----------------------------
# 汇总输出：每窗口明细 + 全局汇总
# -----------------------------
summary_by_window = pd.concat(summary_rows, ignore_index=True) if summary_rows else pd.DataFrame()
group_metrics_all = pd.concat(group_rows, ignore_index=True) if group_rows else pd.DataFrame()

summary_dir = CACHE_ROOT / MODE
summary_dir.mkdir(parents=True, exist_ok=True)

summary_by_window_csv = summary_dir / 'summary_by_window.csv'
summary_aggregate_csv = summary_dir / 'summary_aggregate.csv'
group_metrics_csv = summary_dir / 'group_metrics_all.csv'

if not summary_by_window.empty:
    summary_by_window.to_csv(summary_by_window_csv, index=False)
    summary_aggregate = (
        summary_by_window.groupby('model', as_index=False)
        .agg(
            iv_rmse_mean=('iv_rmse', 'mean'),
            iv_rmse_std=('iv_rmse', 'std'),
            iv_mae_mean=('iv_mae', 'mean'),
            price_rmse_mean=('price_rmse', 'mean'),
            n_windows=('window_id', 'nunique'),
        )
        .sort_values('iv_rmse_mean')
        .reset_index(drop=True)
    )
    summary_aggregate.to_csv(summary_aggregate_csv, index=False)
else:
    summary_aggregate = pd.DataFrame()

if not group_metrics_all.empty:
    group_metrics_all.to_csv(group_metrics_csv, index=False)

print('saved:', summary_by_window_csv)
print('saved:', summary_aggregate_csv)
print('saved:', group_metrics_csv)
summary_aggregate.head(20)

In [ ]:
# -----------------------------
# 可视化1：模型平均 IV RMSE 排名
# -----------------------------
if summary_aggregate.empty:
    print('summary_aggregate is empty.')
else:
    plt.figure(figsize=(8, 4))
    plt.bar(summary_aggregate['model'], summary_aggregate['iv_rmse_mean'])
    plt.title(f'Model Ranking by Mean IV RMSE ({MODE})')
    plt.xlabel('model')
    plt.ylabel('mean iv_rmse')
    plt.xticks(rotation=30)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# -----------------------------
# 可视化2：按 bucket 的分组误差（示例：rbergomi_with_rfsv_prior）
# -----------------------------
if group_metrics_all.empty:
    print('group_metrics_all is empty.')
else:
    gm_plot = group_metrics_all[group_metrics_all.get('group_type', '') == 'maturity_bucket'].copy()
    gm_plot = gm_plot[gm_plot['group'].notna()]
    if gm_plot.empty:
        print('No maturity_bucket group metrics.')
    else:
        agg = (
            gm_plot.groupby('group', as_index=False)
            .agg(iv_rmse=('iv_rmse', 'mean'))
            .sort_values('group')
        )
        plt.figure(figsize=(7, 4))
        plt.bar(agg['group'], agg['iv_rmse'])
        plt.title(f'Rough Group Error by Maturity Bucket ({MODE})')
        plt.xlabel('maturity_bucket')
        plt.ylabel('mean iv_rmse')
        plt.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()
        agg

In [ ]:
# -----------------------------
# speed 模式验收（目标：1小时内）
# -----------------------------
if MODE != 'speed':
    print(f'Current MODE={MODE}; speed SLA check skipped.')
else:
    target_min = 60.0
    print(f'elapsed_min={elapsed_min:.2f}, target_min={target_min:.2f}')
    if elapsed_min <= target_min:
        print('SPEED MODE SLA: PASS (<= 1 hour)')
    else:
        print('SPEED MODE SLA: NOT PASS (> 1 hour)')

    # 粗略估计：若只跑了部分窗口，可按均值外推
    n_done = len(summary_rows)
    n_total = len(windows)
    if n_done > 0 and n_total > 0:
        est_full = elapsed_min * (n_total / n_done)
        print(f'Estimated full-run time: {est_full:.2f} min for {n_total} windows')